## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

#### ARCHITECTURE 4: Multi-Scale Local + Dilated Causal Transformer

The only new architectural idea is the attention pattern:
- local-attention layers model nearby context,
- dilated-attention layers sample farther-back positions,
- the final global causal-attention layer mixes information across the full prefix.

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import shutil

import numpy as np
import torch
from transformers import set_seed

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# Phase-1 files
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# ------------------------------------------------------------
# Reproducibility
# set_seed() seeds Python, NumPy, PyTorch CPU, and PyTorch CUDA.
# IMPORTANT: this runs BEFORE the model is created.
# ------------------------------------------------------------
SEED = 42
set_seed(SEED)

# Training configuration
TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
MODEL_CONTEXT_LENGTH = 2048

# ------------------------------------------------------------
# ARCHITECTURE 4: Multi-Scale Local + Dilated Causal Transformer
# ------------------------------------------------------------
N_EMBD = 600
N_LAYER = 8
N_HEAD = 10
FFN_DIM = 1600
DROPOUT = 0.1

# Each dictionary describes the attention pattern of one Transformer block.
#
# local(window=W):
#   token t attends to the previous W-1 tokens and itself.
#
# dilated(dilation=D):
#   token t attends to t, t-D, t-2D, t-3D, ...
#   This reaches much farther back without using every intermediate position.
#
# global:
#   standard full causal attention over the complete available prefix.
ATTENTION_PATTERNS = [
    {"type": "local",   "value": 256},
    {"type": "dilated", "value": 2},
    {"type": "local",   "value": 512},
    {"type": "dilated", "value": 4},
    {"type": "local",   "value": 512},
    {"type": "dilated", "value": 8},
    {"type": "local",   "value": 1024},
    {"type": "global",  "value": None},
]

assert len(ATTENTION_PATTERNS) == N_LAYER
assert N_EMBD % N_HEAD == 0

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 500
TEMPERATURE = 0.4
TOP_P = 0.95
GENERATION_SEED = SEED

assert MAX_GENERATION_TOKENS < MODEL_CONTEXT_LENGTH

# ------------------------------------------------------------
# Kaggle output
# Keep Trainer checkpoints separate from the final clean model folder.
# ------------------------------------------------------------
OUTPUT_DIR = "/kaggle/working/cs781_phase1_a4_multiscale_dilated"
TRAINER_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "trainer_checkpoints")
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "architecture-4-final")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TRAINER_OUTPUT_DIR, exist_ok=True)

# Hugging Face backup repository name.
HF_REPO_NAME = "cs781-a4-checkpoints"

# Optional restart controls.
# - Local Kaggle restart in the same session: AUTO_RESUME_LOCAL=True is enough.
# - New Kaggle session: set RESUME_FROM_HF_STEP to a previously uploaded step,
#   e.g. 5000, to download checkpoints/checkpoint-5000.zip and resume from it.
AUTO_RESUME_LOCAL = True
RESUME_FROM_HF_STEP = None

print("Configuration loaded.")
print(f"Seed: {SEED}")
print(f"Head dimension: {N_EMBD // N_HEAD}")
print("Attention schedule:")
for i, pattern in enumerate(ATTENTION_PATTERNS, start=1):
    print(f"  Block {i}: {pattern}")


## 4. Load Tokenizer

In [ ]:
from transformers import PreTrainedTokenizerFast
import os

DATA_DIR = "/kaggle/input/datasets/srivenkatm/phase1-simple-data/phase1_simple_data"

TOKENIZER_PATH = os.path.join(DATA_DIR, "tokenizer.json")
TRAIN_BIN = os.path.join(DATA_DIR, "train_tokens.bin")
VAL_BIN = os.path.join(DATA_DIR, "val_tokens.bin")

print("Tokenizer exists:", os.path.exists(TOKENIZER_PATH))
print("Train tokens exist:", os.path.exists(TRAIN_BIN))
print("Val tokens exist:", os.path.exists(VAL_BIN))

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=TOKENIZER_PATH,
    pad_token="<pad>",
    unk_token="<unk>",
    bos_token="<s>",
    eos_token="</s>"
)

print("Vocabulary size:", len(tokenizer))

## 5. Download Training Data From Hugging Face

In [ ]:

# ------------------------------------------------------------
# The tokenizer notebook has ALREADY prepared exactly:
#   200M training token IDs
#   5M validation token IDs
#
# Therefore we do not download and tokenize the raw training JSON again.
# We directly reuse the saved .bin files.
# ------------------------------------------------------------

TRAIN_TOKEN_FILE = os.path.join(DATA_DIR, "train_tokens.bin")
VAL_TOKEN_FILE = os.path.join(DATA_DIR, "val_tokens.bin")

print("Training token file:")
print(TRAIN_TOKEN_FILE)

print("Validation token file:")
print(VAL_TOKEN_FILE)


## 6. Create Train and Validation Tokens Set

In [ ]:

import numpy as np

# ------------------------------------------------------------
# Memory-map the token files.
#
# np.memmap is useful here because 200M token IDs are large.
# It reads small pieces from disk only when they are needed,
# instead of loading the whole file into RAM.
#
# uint16 is enough because our vocabulary is about 50k,
# and 50,000 < 65,536.
# ------------------------------------------------------------

train_tokens = np.memmap(
    TRAIN_TOKEN_FILE,
    dtype=np.uint16,
    mode="r",
    shape=(TRAIN_TOKENS,)
)

val_tokens = np.memmap(
    VAL_TOKEN_FILE,
    dtype=np.uint16,
    mode="r",
    shape=(VAL_TOKENS,)
)

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

# ------------------------------------------------------------
# Safety check 1: uint16 can store token IDs only from 0 to 65,535.
# Therefore the tokenizer vocabulary must fit inside uint16.
# ------------------------------------------------------------
UINT16_MAX_ID = np.iinfo(np.uint16).max
assert len(tokenizer) <= UINT16_MAX_ID + 1, (
    f"Tokenizer vocabulary has {len(tokenizer):,} entries, but uint16 can "
    f"represent at most {UINT16_MAX_ID + 1:,} distinct token IDs."
)
print(f"uint16 vocabulary check passed: vocab_size={len(tokenizer):,}")

# ------------------------------------------------------------
# Safety check 2: scan ALL saved token IDs and verify that every ID
# is valid for the tokenizer currently loaded above.
#
# We scan in chunks so the complete 200M-token file does not need
# to be copied into RAM at once.
# ------------------------------------------------------------
def scan_token_ids(token_array, name, vocab_size, chunk_size=5_000_000):
    min_id = UINT16_MAX_ID
    max_id = 0

    for start in range(0, len(token_array), chunk_size):
        chunk = token_array[start:start + chunk_size]
        if len(chunk) == 0:
            continue

        chunk_min = int(chunk.min())
        chunk_max = int(chunk.max())

        min_id = min(min_id, chunk_min)
        max_id = max(max_id, chunk_max)

    assert min_id >= 0
    assert max_id < vocab_size, (
        f"{name} contains token ID {max_id:,}, but the loaded tokenizer "
        f"has vocabulary size {vocab_size:,}. The .bin files and tokenizer "
        f"do not match."
    )

    print(
        f"{name} full scan passed: "
        f"min_id={min_id:,}, max_id={max_id:,}, vocab_size={vocab_size:,}"
    )


scan_token_ids(train_tokens, "Training tokens", len(tokenizer))
scan_token_ids(val_tokens, "Validation tokens", len(tokenizer))


## 7. Create Token Blocks

In [ ]:

from torch.utils.data import Dataset

class TokenBlockDataset(Dataset):
    """
    Read one fixed-size block directly from the long token file.

    Example with block_size = 4:
        token file: [10, 20, 30, 40, 50, 60, 70, 80]

        block 0 -> [10, 20, 30, 40]
        block 1 -> [50, 60, 70, 80]

    We do not create a huge Python list of all blocks.
    """

    def __init__(self, tokens, block_size):
        self.tokens = tokens
        self.block_size = block_size
        self.num_blocks = len(tokens) // block_size

    def __len__(self):
        return self.num_blocks

    def __getitem__(self, index):
        start = index * self.block_size
        end = start + self.block_size

        # Convert only this block to a normal integer tensor.
        block = np.array(
            self.tokens[start:end],
            dtype=np.int64
        )

        return {
            "input_ids": torch.tensor(
                block,
                dtype=torch.long
            )
        }

print("Token block dataset class created.")


## 8. Create Datasets

In [ ]:

train_dataset = TokenBlockDataset(
    train_tokens,
    TRAIN_BLOCK_SIZE
)

val_dataset = TokenBlockDataset(
    val_tokens,
    TRAIN_BLOCK_SIZE
)

print("Training sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))


## 9. Define The Model

#### ARCHITECTURE 4: Multi-Scale Local + Dilated Causal Transformer

Each layer is an ordinary causal Transformer block. What changes across layers is **which previous positions a token is allowed to attend to**. This makes A4 distinct from A3's fixed sliding-window attention while keeping the new idea easy to explain in viva.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.modeling_outputs import CausalLMOutput

# ------------------------------------------------------------
# ARCHITECTURE 4: MULTI-SCALE LOCAL + DILATED CAUSAL TRANSFORMER
#
# Pipeline:
#   Token embedding + learned position embedding
#       -> local causal-attention block
#       -> dilated causal-attention block
#       -> local causal-attention block
#       -> dilated causal-attention block
#       -> local causal-attention block
#       -> dilated causal-attention block
#       -> wider local causal-attention block
#       -> global causal-attention block
#       -> final LayerNorm
#       -> tied LM head
#
# Key idea:
#   Local attention captures nearby phrases.
#   Dilated attention reaches farther back by attending at intervals.
#   Final global attention lets the last block combine the full prefix.
#
# No pretrained model/config/weights are used.
# ------------------------------------------------------------


def build_pattern_mask(context_length, pattern_type, pattern_value=None):
    """
    Build a boolean causal attention mask of shape [context_length, context_length].

    PyTorch SDPA interprets True as "this query may attend to this key".

    Example for dilation=2 at query position t:
        allowed keys = t, t-2, t-4, t-6, ...
    """
    query_pos = torch.arange(context_length).unsqueeze(1)  # [T, 1]
    key_pos = torch.arange(context_length).unsqueeze(0)    # [1, T]

    distance = query_pos - key_pos
    causal = distance >= 0

    if pattern_type == "global":
        allowed = causal

    elif pattern_type == "local":
        window = int(pattern_value)
        assert window >= 1
        allowed = causal & (distance < window)

    elif pattern_type == "dilated":
        dilation = int(pattern_value)
        assert dilation >= 1
        allowed = causal & ((distance % dilation) == 0)

    else:
        raise ValueError(f"Unknown attention pattern: {pattern_type}")

    return allowed


class PatternCausalSelfAttention(nn.Module):
    """Causal self-attention with a chosen local/dilated/global pattern."""

    def __init__(
        self,
        dim,
        num_heads,
        context_length,
        pattern_type,
        pattern_value=None,
        dropout=0.1,
    ):
        super().__init__()
        assert dim % num_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.context_length = context_length
        self.pattern_type = pattern_type
        self.pattern_value = pattern_value
        self.dropout = dropout

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)

        # The mask is fixed by the architecture, so build it only once.
        # persistent=False means this large mask is NOT written into checkpoints.
        pattern_mask = build_pattern_mask(
            context_length=context_length,
            pattern_type=pattern_type,
            pattern_value=pattern_value,
        )
        self.register_buffer(
            "pattern_mask",
            pattern_mask,
            persistent=False,
        )

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        if past_kv is None:
            # Training / prompt prefill.
            k_all = k
            v_all = v
            key_length = T
            query_start = 0

        else:
            # Incremental generation: append the new token's K/V to the cache.
            assert T == 1, "Cached decoding expects one new token at a time."
            past_k, past_v = past_kv

            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)

            key_length = k_all.size(2)
            query_start = key_length - T

        if key_length > self.context_length:
            raise ValueError(
                f"Attention cache length {key_length} exceeds "
                f"context length {self.context_length}."
            )

        # Slice exactly the query rows and key columns needed in this call.
        # Shape becomes [1, 1, query_length, key_length] and broadcasts over
        # batch and attention heads.
        attn_mask = self.pattern_mask[
            query_start:query_start + T,
            :key_length,
        ][None, None, :, :]

        attn_out = F.scaled_dot_product_attention(
            q,
            k_all,
            v_all,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,  # causality is already encoded in attn_mask
        )

        new_kv = (k_all, v_all) if use_cache else None

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, self.dim)
        )

        return self.o_proj(attn_out), new_kv


class MultiScaleTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        ffn_dim,
        context_length,
        pattern_type,
        pattern_value=None,
        dropout=0.1,
    ):
        super().__init__()

        # Pre-LayerNorm Transformer block.
        self.norm1 = nn.LayerNorm(dim)
        self.attn = PatternCausalSelfAttention(
            dim=dim,
            num_heads=num_heads,
            context_length=context_length,
            pattern_type=pattern_type,
            pattern_value=pattern_value,
            dropout=dropout,
        )

        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, dim),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False):
        attn_out, new_kv = self.attn(
            self.norm1(x),
            past_kv=past_kv,
            use_cache=use_cache,
        )

        # Residual connection around attention.
        x = x + self.dropout(attn_out)

        # Residual connection around feed-forward network.
        x = x + self.dropout(self.ffn(self.norm2(x)))

        return x, new_kv


class MultiScaleDilatedLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_length=2048,
        dim=600,
        num_heads=10,
        ffn_dim=1600,
        attention_patterns=None,
        dropout=0.1,
    ):
        super().__init__()

        if attention_patterns is None:
            raise ValueError("attention_patterns must be provided.")

        self.vocab_size = vocab_size
        self.context_length = context_length
        self.dim = dim
        self.num_heads = num_heads
        self.ffn_dim = ffn_dim
        self.attention_patterns = attention_patterns

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(context_length, dim)
        self.embed_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            MultiScaleTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                context_length=context_length,
                pattern_type=pattern["type"],
                pattern_value=pattern.get("value"),
                dropout=dropout,
            )
            for pattern in attention_patterns
        ])

        self.final_norm = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Initialize first, then tie token-embedding and output weights.
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T = input_ids.shape

        if position_offset + T > self.context_length:
            raise ValueError(
                f"Sequence position {position_offset + T} exceeds "
                f"context length {self.context_length}."
            )

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=input_ids.device,
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )
        x = self.embed_dropout(x)

        if past_key_values is None:
            layer_past = [None] * len(self.blocks)
        else:
            if len(past_key_values) != len(self.blocks):
                raise ValueError("Wrong number of layer caches.")
            layer_past = past_key_values

        new_caches = []

        for block, past_kv in zip(self.blocks, layer_past):
            x, new_kv = block(
                x,
                past_kv=past_kv,
                use_cache=use_cache,
            )

            if use_cache:
                new_caches.append(new_kv)

        x = self.final_norm(x)

        return x, (new_caches if use_cache else None)

    def forward(self, input_ids, labels=None, attention_mask=None, **kwargs):
        # Training examples contain full fixed-length token blocks, so no
        # padding mask is needed here. The architectural causal/sparse mask is
        # handled inside each attention layer.
        hidden, _ = self._run_hidden(input_ids)
        logits = self.lm_head(hidden)

        loss = None

        if labels is not None:
            # Next-token language modelling:
            # position t predicts label at position t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits,
        )

    @torch.inference_mode()
    def prefill(self, input_ids):
        """Process the full prompt once and create K/V caches for all layers."""
        hidden, caches = self._run_hidden(
            input_ids,
            past_key_values=None,
            use_cache=True,
            position_offset=0,
        )

        return self.lm_head(hidden), caches

    @torch.inference_mode()
    def decode_step(
        self,
        input_ids,
        past_key_values,
        position_offset,
    ):
        """Process one newly generated token using the existing K/V caches."""
        hidden, caches = self._run_hidden(
            input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            position_offset=position_offset,
        )

        logits = self.lm_head(hidden[:, -1, :])
        return logits, caches


# ------------------------------------------------------------
# Create Architecture 4.
# set_seed(SEED) was already executed in the configuration cell,
# BEFORE this model is initialized.
# ------------------------------------------------------------
model = MultiScaleDilatedLM(
    vocab_size=len(tokenizer),
    context_length=MODEL_CONTEXT_LENGTH,
    dim=N_EMBD,
    num_heads=N_HEAD,
    ffn_dim=FFN_DIM,
    attention_patterns=ATTENTION_PATTERNS,
    dropout=DROPOUT,
)

model_size = sum(p.numel() for p in model.parameters())

print(f"Model size: {model_size / 1e6:.2f}M parameters")
print(f"Number of Transformer blocks: {len(model.blocks)}")

for i, pattern in enumerate(ATTENTION_PATTERNS, start=1):
    print(f"Block {i}: {pattern}")

# Phase-1 rule: model must be between 50M and 300M parameters.
assert 50_000_000 <= model_size <= 300_000_000


In [ ]:

def data_collator(features):
    """
    Stack equal-length token blocks into one batch.

    For causal language modelling:
        input_ids = [t0, t1, t2, ...]
        labels    = [t0, t1, t2, ...]

    The model internally shifts them by one position when computing
    next-token prediction loss.
    """
    input_ids = torch.stack([
        item["input_ids"]
        for item in features
    ])

    return {
        "input_ids": input_ids,
        "labels": input_ids.clone()
    }


## 10. Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=TRAINER_OUTPUT_DIR,
    num_train_epochs=EPOCHS,

    # Physical batch = 1; accumulate 8 batches before one optimizer update.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=500,
    lr_scheduler_type="cosine",

    # Mixed precision on Kaggle GPU.
    fp16=torch.cuda.is_available(),

    # Evaluate and save at the SAME interval so Trainer can restore
    # the checkpoint having the lowest validation loss.
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    logging_strategy="steps",
    logging_steps=100,

    # Keep only a few checkpoints locally. Every saved checkpoint is also
    # zipped and uploaded to the private Hugging Face backup repo below.
    save_total_limit=2,
    save_only_model=False,

    # Automatically restore the lowest-validation-loss checkpoint at the end.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # IMPORTANT for our tied token-embedding / LM-head weights.
    # Standard safetensors saving can reject shared storage.
    save_safetensors=False,

    # Reproducibility.
    seed=SEED,
    data_seed=SEED,

    report_to="none",
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
)


## 11. Training

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from transformers import Trainer, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint

# ------------------------------------------------------------
# PRIVATE HUGGING FACE CHECKPOINT BACKUP
# ------------------------------------------------------------
hf_api = HfApi(token=HF_TOKEN)
HF_USERNAME = hf_api.whoami()["name"]
HF_CHECKPOINT_REPO = f"{HF_USERNAME}/{HF_REPO_NAME}"

hf_api.create_repo(
    repo_id=HF_CHECKPOINT_REPO,
    repo_type="model",
    private=True,
    exist_ok=True,
)

print("Private Hugging Face backup repo:")
print(HF_CHECKPOINT_REPO)


class HuggingFaceCheckpointBackup(TrainerCallback):
    """
    Whenever Trainer saves checkpoint-N locally:
      1. zip the complete checkpoint directory,
      2. upload it to checkpoints/checkpoint-N.zip on Hugging Face,
      3. delete only the temporary local ZIP.

    The original checkpoint directory remains available for local resume until
    Trainer removes old checkpoints according to save_total_limit.
    """

    def on_save(self, args, state, control, **kwargs):
        if not state.is_world_process_zero:
            return control

        checkpoint_name = f"checkpoint-{state.global_step}"
        checkpoint_dir = os.path.join(args.output_dir, checkpoint_name)

        if not os.path.isdir(checkpoint_dir):
            print(f"Checkpoint folder not found: {checkpoint_dir}")
            return control

        zip_base = os.path.join(
            OUTPUT_DIR,
            f"_upload_{checkpoint_name}",
        )
        zip_path = zip_base + ".zip"

        try:
            # root_dir=checkpoint_dir puts the checkpoint CONTENTS at the root
            # of the zip, so it can later be unpacked directly into
            # trainer_checkpoints/checkpoint-N/.
            shutil.make_archive(
                zip_base,
                "zip",
                root_dir=checkpoint_dir,
            )

            hf_api.upload_file(
                path_or_fileobj=zip_path,
                path_in_repo=f"checkpoints/{checkpoint_name}.zip",
                repo_id=HF_CHECKPOINT_REPO,
                repo_type="model",
                commit_message=f"Backup {checkpoint_name}",
            )

            print(f"Uploaded {checkpoint_name} to Hugging Face.")

        except Exception as e:
            # Do not kill a long training run only because a network upload
            # temporarily failed.
            print(f"WARNING: could not upload {checkpoint_name}: {e}")

        finally:
            if os.path.exists(zip_path):
                os.remove(zip_path)

        return control


# ------------------------------------------------------------
# OPTIONAL RESUME LOGIC
# ------------------------------------------------------------
resume_checkpoint = None

# New Kaggle session: recover one previously uploaded HF checkpoint.
if RESUME_FROM_HF_STEP is not None:
    checkpoint_name = f"checkpoint-{RESUME_FROM_HF_STEP}"

    remote_zip = hf_hub_download(
        repo_id=HF_CHECKPOINT_REPO,
        filename=f"checkpoints/{checkpoint_name}.zip",
        repo_type="model",
        token=HF_TOKEN,
    )

    recovered_dir = os.path.join(
        TRAINER_OUTPUT_DIR,
        checkpoint_name,
    )

    # Remove a partial old extraction, if one exists.
    if os.path.isdir(recovered_dir):
        shutil.rmtree(recovered_dir)

    os.makedirs(recovered_dir, exist_ok=True)
    shutil.unpack_archive(remote_zip, recovered_dir)

    resume_checkpoint = recovered_dir
    print("Recovered checkpoint from Hugging Face:")
    print(resume_checkpoint)

# Same Kaggle session: automatically continue from newest local checkpoint.
elif AUTO_RESUME_LOCAL:
    local_last_checkpoint = get_last_checkpoint(TRAINER_OUTPUT_DIR)

    if local_last_checkpoint is not None:
        resume_checkpoint = local_last_checkpoint
        print("Found local checkpoint:")
        print(resume_checkpoint)


# ------------------------------------------------------------
# TRAINER
# ------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[HuggingFaceCheckpointBackup()],
)

# Remember the checkpoint step before training so throughput is still meaningful
# when this notebook resumes an interrupted run.
initial_global_step = 0
if resume_checkpoint is not None:
    try:
        initial_global_step = int(
            os.path.basename(resume_checkpoint).split("-")[-1]
        )
    except Exception:
        initial_global_step = 0

start_time = time.time()

trainer.train(
    resume_from_checkpoint=resume_checkpoint
)

training_time = time.time() - start_time

updates_this_run = max(
    0,
    trainer.state.global_step - initial_global_step,
)

approx_tokens_this_run = (
    updates_this_run
    * training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
    * TRAIN_BLOCK_SIZE
)

tokens_per_second = (
    approx_tokens_this_run / training_time
    if training_time > 0
    else 0.0
)

tokens_per_epoch = len(train_dataset) * TRAIN_BLOCK_SIZE

print(f"Training time this run: {training_time / 3600:.2f} hours")
print(f"Optimizer updates this run: {updates_this_run:,}")
print(f"Approx. token throughput this run: {tokens_per_second:,.0f} tokens/sec")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

# Because load_best_model_at_end=True, trainer.model now contains the
# best-validation-loss checkpoint rather than blindly using the last step.


## 12. Perplexity

In [ ]:
# trainer.model is already the best checkpoint because
# load_best_model_at_end=True.
eval_results = trainer.evaluate()
best_eval_loss = eval_results["eval_loss"]

try:
    perplexity = math.exp(best_eval_loss)
except OverflowError:
    perplexity = float("inf")

print(f"Best-checkpoint validation loss: {best_eval_loss:.6f}")
print(f"Best-checkpoint validation perplexity: {perplexity:.4f}")
print("Selected checkpoint:", trainer.state.best_model_checkpoint)


## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps = []
train_losses = []
eval_steps = []
eval_losses = []

for log in logs:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])

    if "eval_loss" in log and "step" in log:
        eval_steps.append(log["step"])
        eval_losses.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label="Training Loss")
plt.plot(eval_steps, eval_losses, label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Architecture 4: Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:")
print(LOSS_PLOT_PATH)


## 14. Save Final Model to Kaggle Output

In [ ]:
# ------------------------------------------------------------
# Save the BEST validation-loss model, tokenizer, architecture config,
# and training summary into a clean final-model folder.
# ------------------------------------------------------------
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

best_model = trainer.model

# Custom PyTorch model weights.
torch.save(
    best_model.state_dict(),
    os.path.join(MODEL_OUTPUT_DIR, "model_state.pt"),
)

# Everything needed to reconstruct Architecture 4 exactly.
architecture_config = {
    "architecture": "MultiScaleDilatedLM",
    "vocab_size": len(tokenizer),
    "context_length": MODEL_CONTEXT_LENGTH,
    "dim": N_EMBD,
    "num_heads": N_HEAD,
    "ffn_dim": FFN_DIM,
    "dropout": DROPOUT,
    "attention_patterns": ATTENTION_PATTERNS,
    "seed": SEED,
}

with open(
    os.path.join(MODEL_OUTPUT_DIR, "architecture_config.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(architecture_config, f, indent=2)

# Evidence useful for the final report / viva.
training_summary = {
    "best_checkpoint": (
        os.path.basename(trainer.state.best_model_checkpoint)
        if trainer.state.best_model_checkpoint is not None
        else None
    ),
    "best_eval_loss": (
        float(trainer.state.best_metric)
        if trainer.state.best_metric is not None
        else None
    ),
    "final_best_model_eval_loss": float(best_eval_loss),
    "final_best_model_perplexity": float(perplexity),
    "training_time_this_run_hours": float(training_time / 3600),
    "nominal_training_tokens_per_epoch": int(tokens_per_epoch),
    "seed": SEED,
}

with open(
    os.path.join(MODEL_OUTPUT_DIR, "training_summary.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(training_summary, f, indent=2)

# Reuse exactly the same tokenizer for all architectures.
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("Best model, config, tokenizer, and summary saved to:")
print(MODEL_OUTPUT_DIR)


## 15. Push Model to Hugging Face Hub

In [ ]:
# ------------------------------------------------------------
# Upload the CLEAN final model folder to the same private HF repo used for
# checkpoint backups. Checkpoint ZIPs remain under checkpoints/.
# ------------------------------------------------------------
hf_api.upload_folder(
    folder_path=MODEL_OUTPUT_DIR,
    path_in_repo="final_model",
    repo_id=HF_CHECKPOINT_REPO,
    repo_type="model",
    commit_message="Upload final Architecture 4 model",
)

print("Final model uploaded to:")
print(f"{HF_CHECKPOINT_REPO}/tree/main/final_model")


## 16. Load Test Data from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

# ------------------------------------------------------------
# Check whether reserving MAX_GENERATION_TOKENS would truncate any official
# prompt. This does not modify the data; it only reports what generation will do.
# ------------------------------------------------------------
max_prompt_tokens = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS

prompt_token_lengths = {
    str(example_id): len(
        tokenizer.encode(
            record["text"],
            add_special_tokens=False,
        )
    )
    for example_id, record in test_data.items()
}

length_values = list(prompt_token_lengths.values())

too_long_ids = [
    example_id
    for example_id, length in prompt_token_lengths.items()
    if length > max_prompt_tokens
]

print(f"Maximum prompt tokens without truncation: {max_prompt_tokens}")
print(f"Shortest test prompt: {min(length_values)} tokens")
print(f"Longest test prompt: {max(length_values)} tokens")
print(f"Prompts that would be left-truncated: {len(too_long_ids)}")

if too_long_ids:
    print("WARNING: some prompts will be left-truncated during generation.")
    print("First affected IDs:", too_long_ids[:10])
else:
    print("Good: no test prompt will be truncated.")


## 18. Load Final Model for Generation

In [ ]:
from transformers import PreTrainedTokenizerFast
import torch

# ------------------------------------------------------------
# Reload the final saved Architecture 4 exactly as it will be used later.
# ------------------------------------------------------------
generation_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_OUTPUT_DIR
)

with open(
    os.path.join(MODEL_OUTPUT_DIR, "architecture_config.json"),
    "r",
    encoding="utf-8",
) as f:
    saved_config = json.load(f)


generation_model = MultiScaleDilatedLM(
    vocab_size=len(generation_tokenizer),
    context_length=saved_config["context_length"],
    dim=saved_config["dim"],
    num_heads=saved_config["num_heads"],
    ffn_dim=saved_config["ffn_dim"],
    attention_patterns=saved_config["attention_patterns"],
    dropout=saved_config["dropout"],
)

state_dict = torch.load(
    os.path.join(MODEL_OUTPUT_DIR, "model_state.pt"),
    map_location="cpu",
    weights_only=True,
)

generation_model.load_state_dict(state_dict)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

generation_model = generation_model.to(device)
generation_model.eval()

print("Device:", device)
print(
    f"Parameters: "
    f"{sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M"
)


## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
def sample_top_p(logits, temperature=0.4, top_p=0.95):
    """Sample one token using temperature + nucleus (top-p) sampling."""
    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)

    sorted_probs, sorted_ids = torch.sort(
        probs,
        descending=True,
    )

    cumulative = torch.cumsum(
        sorted_probs,
        dim=-1,
    )

    remove = cumulative > top_p
    remove[:, 1:] = remove[:, :-1].clone()
    remove[:, 0] = False

    sorted_probs = sorted_probs.masked_fill(
        remove,
        0.0,
    )

    sorted_probs = (
        sorted_probs
        / sorted_probs.sum(dim=-1, keepdim=True)
    )

    sampled_index = torch.multinomial(
        sorted_probs,
        1,
    )

    return sorted_ids.gather(
        1,
        sampled_index,
    )


def generate_prompts(
    prompts,
    model,
    tokenizer,
    device,
    target_words=128,
):
    """
    Generate one continuation per prompt.

    The full prompt is processed once with model.prefill().
    Afterwards every Transformer layer reuses its K/V cache, so generation
    processes only one new token at a time.
    """
    predictions = []

    for prompt in prompts:
        prompt_ids = tokenizer.encode(
            prompt,
            add_special_tokens=False,
        )

        # Reserve enough context positions for generation.
        max_prompt_tokens = (
            model.context_length
            - MAX_GENERATION_TOKENS
        )

        if len(prompt_ids) > max_prompt_tokens:
            prompt_ids = prompt_ids[-max_prompt_tokens:]

        if len(prompt_ids) == 0:
            raise ValueError("Encountered an empty tokenized prompt.")

        x = torch.tensor(
            [prompt_ids],
            dtype=torch.long,
            device=device,
        )

        generated_ids = []

        with torch.inference_mode():
            # Prompt positions are 0 ... len(prompt_ids)-1.
            prompt_logits, past_key_values = model.prefill(x)
            logits = prompt_logits[:, -1, :]

            for step in range(MAX_GENERATION_TOKENS):
                next_token = sample_top_p(
                    logits,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                )

                generated_ids.append(
                    int(next_token.item())
                )

                generated_text = tokenizer.decode(
                    generated_ids,
                    skip_special_tokens=True,
                )

                # Competition requirement is 128 whitespace-separated WORDS,
                # not 128 tokenizer tokens.
                if len(generated_text.split()) >= target_words:
                    break

                # First generated token is placed at position len(prompt_ids).
                position = len(prompt_ids) + step

                logits, past_key_values = model.decode_step(
                    next_token,
                    past_key_values=past_key_values,
                    position_offset=position,
                )

        words = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ).split()

        if len(words) < target_words:
            raise RuntimeError(
                f"Generation produced only {len(words)} words before the "
                f"{MAX_GENERATION_TOKENS}-token limit. "
                "Increase MAX_GENERATION_TOKENS and re-check prompt truncation."
            )

        predictions.append(
            " ".join(words[:target_words])
        )

    return predictions


## 20. Generate Predictions for the Test Set

In [ ]:
# Reset RNG immediately before stochastic decoding.
# Training consumes RNG state, so this makes generation reproducible for a
# fixed saved model + fixed decoding configuration.
set_seed(GENERATION_SEED)

predictions = {}

# This groups prompts for progress reporting. generate_prompts() intentionally
# processes them one-by-one because prompt lengths vary and every sequence has
# its own growing K/V cache.
GENERATION_GROUP_SIZE = 32

items = list(test_data.items())
total_examples = len(items)

print(f"Generation seed: {GENERATION_SEED}")
print(f"Total test examples: {total_examples}")

for start in range(
    0,
    total_examples,
    GENERATION_GROUP_SIZE,
):
    group = items[
        start:start + GENERATION_GROUP_SIZE
    ]

    group_ids = [
        str(example_id)
        for example_id, _ in group
    ]

    group_prompts = [
        record["text"]
        for _, record in group
    ]

    group_predictions = generate_prompts(
        prompts=group_prompts,
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=TARGET_GENERATION_WORDS,
    )

    for example_id, prediction in zip(
        group_ids,
        group_predictions,
    ):
        predictions[example_id] = {
            "text": prediction
        }

    completed = start + len(group)

    print(
        f"Completed {completed}/{total_examples} "
        f"({100 * completed / total_examples:.1f}%)"
    )


## 21. Save prediction.json

In [ ]:
# ------------------------------------------------------------
# Validate the complete Phase-1 submission BEFORE writing prediction.json.
# Required by the capstone:
#   - every official test ID must be present
#   - no unknown/extra IDs
#   - every text must be non-empty
#   - every continuation must contain exactly 128 whitespace-separated words
# ------------------------------------------------------------
official_ids = {str(example_id) for example_id in test_data.keys()}
predicted_ids = set(predictions.keys())

missing_ids = official_ids - predicted_ids
extra_ids = predicted_ids - official_ids

assert not missing_ids, f"Missing official test IDs: {sorted(missing_ids)[:10]}"
assert not extra_ids, f"Unknown/extra prediction IDs: {sorted(extra_ids)[:10]}"

for example_id in official_ids:
    record = predictions[example_id]

    assert isinstance(record, dict), (
        f"Prediction for ID {example_id} must be an object/dict."
    )
    assert "text" in record, f"Prediction for ID {example_id} has no 'text' field."

    text = record["text"]
    assert isinstance(text, str) and text.strip(), (
        f"Prediction text for ID {example_id} is empty."
    )

    word_count = len(text.split())
    assert word_count == TARGET_GENERATION_WORDS, (
        f"ID {example_id} has {word_count} whitespace-separated words; "
        f"expected exactly {TARGET_GENERATION_WORDS}."
    )

print(
    f"Submission validation passed: {len(predictions)} IDs, "
    f"all continuations have exactly {TARGET_GENERATION_WORDS} words."
)

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
# ------------------------------------------------------------
# Create Codabench submission.zip with prediction.json at ZIP ROOT.
# ------------------------------------------------------------
with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zipf:
    zipf.write(
        PREDICTION_PATH,
        arcname="prediction.json",
    )

print("Codabench submission created:")
print(SUBMISSION_ZIP)

# ------------------------------------------------------------
# Upload final prediction/submission artifacts to private HF backup repo.
# ------------------------------------------------------------
for local_path, remote_path in [
    (PREDICTION_PATH, "final_outputs/prediction.json"),
    (SUBMISSION_ZIP, "final_outputs/submission.zip"),
    (LOSS_PLOT_PATH, "final_outputs/training_validation_loss.png"),
]:
    hf_api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=remote_path,
        repo_id=HF_CHECKPOINT_REPO,
        repo_type="model",
        commit_message=f"Upload {os.path.basename(local_path)}",
    )

# ------------------------------------------------------------
# Complete backup of OUTPUT_DIR.
# Put the backup ZIP OUTSIDE OUTPUT_DIR so it does not zip itself recursively.
# Older local checkpoints removed by save_total_limit are already preserved as
# individual ZIP files in checkpoints/ on Hugging Face.
# ------------------------------------------------------------
FINAL_BACKUP_BASE = "/kaggle/working/CS781_A4_COMPLETE_BACKUP"
FINAL_BACKUP_ZIP = FINAL_BACKUP_BASE + ".zip"

shutil.make_archive(
    FINAL_BACKUP_BASE,
    "zip",
    root_dir=OUTPUT_DIR,
)

hf_api.upload_file(
    path_or_fileobj=FINAL_BACKUP_ZIP,
    path_in_repo="CS781_A4_COMPLETE_BACKUP.zip",
    repo_id=HF_CHECKPOINT_REPO,
    repo_type="model",
    commit_message="Upload complete Architecture 4 backup",
)

print("Final outputs and complete backup uploaded to:")
print(HF_CHECKPOINT_REPO)
print("Complete backup:")
print(FINAL_BACKUP_ZIP)
